# 03 · 实验跟踪与配置管理  Experiment Tracking & Config Management

> **能力标签**：GA4（训练优化 / Training Optimization）

## 目标 Objectives

完成本课后，你应该能够：

1. 描述**实验跟踪**（experiment tracking）的作用：记录超参数、指标、模型权重，对比多次实验。
2. 说明 **Weights & Biases (W&B)** 和 **CSVLogger** 的基本用法（prose 形式，不导入包）。
3. 实现**检查点保存与恢复**（checkpoint save/load），对应 `w10-checkpoint` 练习。
4. 实现基于 **`argparse`** 的 CLI 配置，并演示**环境变量覆盖**（env-var override）模式。

> 对应能力：**GA4**

## 概念讲解 Concepts

### 1. 实验跟踪 Experiment Tracking

每次训练都是一次**实验**（experiment）。不跟踪就容易出现：
- 不知道哪个 checkpoint 对应哪组超参数
- 调参时无法对比不同配置的学习曲线

**两种常用工具**：

| 工具 | 类型 | 特点 |
|------|------|------|
| Weights & Biases (W&B) | 云端服务 | 漂亮的 UI、团队协作、自动日志 |
| CSVLogger (Lightning/自定义) | 本地文件 | 零依赖、离线可用、易解析 |

---

### 2. W&B 基本用法（Prose + 伪代码）

```python
# 伪代码（需要 wandb 包和账号）
import wandb

wandb.init(project="my-project", config={"lr": 1e-3, "epochs": 20})

for epoch in range(20):
    loss = train_one_epoch(...)
    wandb.log({"epoch": epoch, "train_loss": loss})   # 自动上传到 W&B

wandb.finish()
```

在 **PyTorch Lightning** 中：

```python
# 伪代码
from lightning.pytorch.loggers import WandbLogger
logger = WandbLogger(project="my-project")
trainer = pl.Trainer(logger=logger)
```

Lightning 会自动记录 `self.log(...)` 里的所有指标，无需额外代码。

---

### 3. CSVLogger（本地日志）

```python
# 伪代码
import csv, pathlib

class CSVLogger:
    def __init__(self, path):
        self.path = pathlib.Path(path)
        self._file = open(self.path, "w", newline="")
        self._writer = None

    def log(self, **kwargs):
        if self._writer is None:
            self._writer = csv.DictWriter(self._file, fieldnames=list(kwargs))
            self._writer.writeheader()
        self._writer.writerow(kwargs)
        self._file.flush()
```

---

### 4. Checkpoint 保存与恢复

**检查点**（checkpoint）= 训练状态的快照，包含：
- 模型权重（`model.state_dict()`）
- 优化器状态（`optimizer.state_dict()`，含动量、Adam 二阶矩）
- 当前步数/epoch（用于断点续训）

断点续训（resume）只需 `torch.load` 后恢复 `state_dict`，从保存的 step 继续循环。

---

### 5. Config-via-CLI：argparse + 环境变量

工程训练脚本通常有**多层配置**：

```
默认值 (defaults) < 配置文件 (config.yaml) < CLI 参数 (--lr 0.001) < 环境变量 (LR=0.0001)
```

环境变量覆盖（env-var override）模式：

```python
import os, argparse

parser = argparse.ArgumentParser()
parser.add_argument("--lr", type=float, default=1e-3)
args = parser.parse_args([])

# 允许用环境变量覆盖 Allow env-var override
args.lr = float(os.environ.get("LR", args.lr))
```

这在 Docker/Kubernetes 部署时非常实用：不修改代码，只改环境变量。


## 示例 Worked Example

In [ ]:
# Setup — 自包含，CPU，可复现
import tempfile, os, csv, pathlib
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
DEVICE = torch.device("cpu")
TMPDIR = tempfile.mkdtemp()
print("PyTorch:", torch.__version__)
print("Temp dir:", TMPDIR)

In [ ]:
# 1. Checkpoint 保存与恢复 (mirrors w10-checkpoint)

def save_checkpoint(model, optimizer, step, path):
    '''保存模型/优化器状态与步数到 path。
    Save model/optimizer state and step to path.
    mirrors w10-checkpoint exercise solution.
    '''
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "step": step,
    }, path)


def load_checkpoint(model, optimizer, path):
    '''从 path 恢复模型/优化器状态，返回 step。
    Restore model/optimizer state from path, return step.
    '''
    ckpt = torch.load(path, weights_only=True)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    return ckpt["step"]


print("checkpoint 函数定义完成 checkpoint functions defined")

In [ ]:
# 2. 训练 + checkpoint 往返测试 Training + checkpoint roundtrip

torch.manual_seed(0)
model = nn.Linear(4, 3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 运行一步以产生 Adam 状态 Run one step to populate Adam state
loss = model(torch.randn(5, 4)).sum(); loss.backward(); optimizer.step()
optimizer.zero_grad()

ckpt_path = pathlib.Path(TMPDIR) / "model_step042.pt"
save_checkpoint(model, optimizer, step=42, path=ckpt_path)
print(f"Checkpoint 保存到 saved to: {ckpt_path}")

# 恢复到新模型 Restore to fresh model
model2 = nn.Linear(4, 3)
opt2 = torch.optim.Adam(model2.parameters(), lr=1e-3)
step_loaded = load_checkpoint(model2, opt2, ckpt_path)

print(f"恢复的步数 Loaded step: {step_loaded}")
assert step_loaded == 42

# 验证权重完全一致 Verify weights are identical
for (n1, p1), (n2, p2) in zip(model.named_parameters(), model2.named_parameters()):
    assert torch.allclose(p1, p2), f"{n1}: weights differ after restore"

print("\u2713 Checkpoint 往返测试通过 roundtrip passed")

In [ ]:
# 3. 简易 CSVLogger

class CSVLogger:
    '''把训练指标写入 CSV 文件的简易日志器。
    Simple logger that writes training metrics to a CSV file.
    '''
    def __init__(self, path):
        self.path = pathlib.Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._file = open(self.path, "w", newline="", encoding="utf-8")
        self._writer = None

    def log(self, **kwargs):
        if self._writer is None:
            self._writer = csv.DictWriter(self._file, fieldnames=list(kwargs))
            self._writer.writeheader()
        self._writer.writerow(kwargs)
        self._file.flush()

    def close(self):
        self._file.close()


# 使用示例 Usage example
log_path = pathlib.Path(TMPDIR) / "train_log.csv"
logger = CSVLogger(log_path)

for epoch in range(5):
    fake_loss = 1.0 / (epoch + 1)
    logger.log(epoch=epoch, train_loss=round(fake_loss, 4))

logger.close()

# 验证 CSV 内容 Verify CSV content
with open(log_path, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(f"记录行数 rows written: {len(rows)}")
for r in rows:
    print(f"  epoch={r['epoch']}  train_loss={r['train_loss']}")

assert len(rows) == 5
assert rows[0]["epoch"] == "0"
print("\u2713 CSVLogger 通过 passed")

In [ ]:
# 4. argparse + 环境变量覆盖 CLI config + env-var override

import argparse, os

def make_parser():
    p = argparse.ArgumentParser(description="训练配置示例 Training config demo")
    p.add_argument("--lr", type=float, default=1e-3, help="学习率 learning rate")
    p.add_argument("--batch_size", type=int, default=32, help="批次大小 batch size")
    p.add_argument("--max_epochs", type=int, default=10, help="最大 epoch 数")
    p.add_argument("--use_wandb", action="store_true", help="启用 W&B 日志")
    return p

# 在 notebook 中用 [] 代替命令行 Use [] in notebook (no sys.argv)
parser = make_parser()
args = parser.parse_args([])

# 环境变量覆盖（优先级最高）env-var override (highest priority)
args.lr = float(os.environ.get("LR", args.lr))
args.batch_size = int(os.environ.get("BATCH_SIZE", args.batch_size))
args.max_epochs = int(os.environ.get("MAX_EPOCHS", args.max_epochs))

print("配置 Config:")
print(f"  lr         = {args.lr}")
print(f"  batch_size = {args.batch_size}")
print(f"  max_epochs = {args.max_epochs}")
print(f"  use_wandb  = {args.use_wandb}")

# 演示 env-var 覆盖 Demonstrate env-var override
os.environ["LR"] = "5e-4"
args2 = parser.parse_args([])
args2.lr = float(os.environ.get("LR", args2.lr))
print(f"\n设置 LR=5e-4 后 After setting LR=5e-4: args.lr = {args2.lr}")
assert abs(args2.lr - 5e-4) < 1e-9

# 清理 Cleanup
del os.environ["LR"]
print("\u2713 argparse + env-var override 通过 passed")

In [ ]:
# 5. 完整训练循环 + checkpoint + logging

torch.manual_seed(42)

# 合成数据 Synthetic data
N = 100
X_tr = torch.randn(N, 4)
w_true = torch.tensor([1.0, -0.5, 0.3, 0.8])
y_tr = (X_tr @ w_true).unsqueeze(1) + 0.1 * torch.randn(N, 1)

model3 = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 1))
opt3 = torch.optim.Adam(model3.parameters(), lr=1e-3)

log_path2 = pathlib.Path(TMPDIR) / "full_train.csv"
logger2 = CSVLogger(log_path2)
ckpt_dir = pathlib.Path(TMPDIR) / "checkpoints"
ckpt_dir.mkdir(exist_ok=True)

NUM_EPOCHS = 15
CKPT_EVERY = 5   # 每 5 个 epoch 保存一次

for epoch in range(NUM_EPOCHS):
    model3.train()
    perm = torch.randperm(N)
    epoch_loss = 0.0; n_batches = 0
    for s in range(0, N, 16):
        idx = perm[s:s+16]
        xb, yb = X_tr[idx], y_tr[idx]
        opt3.zero_grad()
        loss = F.mse_loss(model3(xb), yb)
        loss.backward(); opt3.step()
        epoch_loss += loss.item(); n_batches += 1

    avg = epoch_loss / n_batches
    logger2.log(epoch=epoch, train_loss=round(avg, 6))

    if (epoch + 1) % CKPT_EVERY == 0:
        ckpt_p = ckpt_dir / f"step_{epoch+1:04d}.pt"
        save_checkpoint(model3, opt3, step=epoch+1, path=ckpt_p)
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS}  loss={avg:.4f}  ckpt saved")

logger2.close()

# 验证最后一个 checkpoint 可以恢复
last_ckpt = ckpt_dir / f"step_{NUM_EPOCHS:04d}.pt"
model_r = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 1))
opt_r = torch.optim.Adam(model_r.parameters())
step_r = load_checkpoint(model_r, opt_r, last_ckpt)
assert step_r == NUM_EPOCHS
print(f"\n\u2713 最终 checkpoint 恢复成功 Final ckpt restored at step={step_r}")

## 动手 Exercise

在下面的 cell 中，**实现 `checkpoint_and_resume`**：

- 接受 `model`、`optimizer`、`step`、`ckpt_dir`（`pathlib.Path`）参数
- 文件名格式：`ckpt_dir / f"step_{step:06d}.pt"`
- 调用 `save_checkpoint` 保存
- 然后用 `load_checkpoint` 恢复到一个全新的同架构模型，返回恢复的 step

这个练习帮助你理解"保存 → 创建新模型 → 加载"的完整流程。

In [ ]:
def checkpoint_and_resume(model, optimizer, step, ckpt_dir):
    '''保存检查点，然后从新模型恢复，返回恢复的步数。
    Save checkpoint, then restore to a fresh model of same arch, return loaded step.
    '''
    # TODO: 实现
    # Hint:
    #   path = ckpt_dir / f"step_{step:06d}.pt"
    #   save_checkpoint(model, optimizer, step, path)
    #   # 创建同架构新模型
    #   model_fresh = type(model)(...)
    #   opt_fresh = type(optimizer)(model_fresh.parameters(), lr=...)
    #   step_loaded = load_checkpoint(model_fresh, opt_fresh, path)
    #   return step_loaded
    raise NotImplementedError("请实现 checkpoint_and_resume")

# 实现后取消注释
# import pathlib
# ckpt_dir_ex = pathlib.Path(TMPDIR) / "ex_ckpts"
# ckpt_dir_ex.mkdir(exist_ok=True)
# m_ex = nn.Linear(3, 1)
# o_ex = torch.optim.SGD(m_ex.parameters(), lr=0.1)
# result_step = checkpoint_and_resume(m_ex, o_ex, step=99, ckpt_dir=ckpt_dir_ex)
# assert result_step == 99
# print(f"\u2713 checkpoint_and_resume 通过 passed, step={result_step}")
print("提示：实现 checkpoint_and_resume 后取消注释运行验证")

## 延伸阅读 Further Reading

1. **Weights & Biases 快速入门**：<https://docs.wandb.ai/quickstart> — 5 分钟上手 W&B 实验跟踪。
2. **MLflow**：开源实验跟踪替代方案，<https://mlflow.org/>，支持本地部署。
3. **Hydra**（Facebook Research）：<https://hydra.cc/> — 比 argparse 更强大的配置框架，支持 YAML 组合与参数扫描。
4. **PyTorch `torch.save` / `torch.load` 文档**：<https://pytorch.org/docs/stable/generated/torch.save.html> — `weights_only=True` 参数（安全性相关）。
5. **Lightning ModelCheckpoint Callback**：<https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.callbacks.ModelCheckpoint.html> — 生产级别的自动 checkpoint 管理。
6. **Argparse 官方文档**：<https://docs.python.org/3/library/argparse.html> — `action="store_true"`、`nargs`、`type` 等高级用法。

## 关联练习 Related Assignments

```bash
python check.py 02-checkpoint
```

> 实现 `save_checkpoint(model, optimizer, step, path)` 和 `load_checkpoint(model, optimizer, path)`。
> 关键：`torch.save({"model": ..., "optimizer": ..., "step": ...}, path)`；`torch.load(..., weights_only=True)` 后调用 `load_state_dict`；返回 `ckpt["step"]`。

> 能力标签：**GA4** · threshold ≥ 0.7